# Capítulo 5 · Proyecto 2 · Clasificación de celulares

> ¿Qué características de un celular necesitas para predecir su gama de precio y qué modelo recomendarías a una tienda?

Usa los 2000 equipos etiquetados del archivo `train.csv`. La copia local está en `workspace/chapter_05/datos/mobile_price_train.csv` desde la raíz del repositorio. Documenta tu trabajo de modo que otra persona pueda repetirlo. [Mobile Price Classification en Kaggle](https://www.kaggle.com/datasets/iabhishekofficial/mobile-price-classification)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

In [ ]:
raiz = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
ruta_datos = raiz / "workspace" / "chapter_05" / "datos" / "mobile_price_train.csv"
df = pd.read_csv(ruta_datos)
df.head()

In [ ]:
df.shape

## Etapa 1 · Preparación y selección de características

1. Identifica las 20 características y separa la variable objetivo `price_range`. Sus cuatro valores van de 0 (gama más baja) a 3 (gama más alta).
2. Reserva el 20 % para prueba con una división estratificada y `random_state=42`. En el 80 % de entrenamiento, revisa los valores faltantes, los duplicados y la distribución de las gamas. Justifica las decisiones de limpieza.
3. Antes de entrenar, anota tres características que esperas que ayuden a distinguir las gamas. Elige el algoritmo de selección que prefieras y explica qué criterio utiliza para conservar una característica.

### 1.1 Características y variable objetivo

In [ ]:
X = df.drop(columns="price_range")
y = df["price_range"]

In [ ]:
print(len(X.columns), "características")
X.columns.tolist()

**Qué veo.** Son 20 características. Seis son de sí o no, con 0 o 1 (`blue`, `dual_sim`, `four_g`, `three_g`, `touch_screen` y `wifi`), y las demás son números: batería, memoria, cámaras, pantalla, peso y procesador. `price_range` es lo que quiero predecir.

### 1.2 Entrenamiento y prueba: 80-20

Separo el 20% para prueba. `stratify=y` mantiene la misma proporción de las cuatro gamas y `random_state=42` permite repetir la división. Todo lo que sigue se revisa solo en el 80% de entrenamiento.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE,
)

In [ ]:
print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)

Faltantes

In [ ]:
X_train.isna().sum().sum() #total de nulos

Filas duplicadas

In [ ]:
X_train.duplicated().sum()

Distribución de las gamas

In [ ]:
y_train.value_counts().sort_index()

In [ ]:
fig, ax = plt.subplots()

sns.countplot(x=y_train, ax=ax)

ax.set_title("Celulares por gama de precio (entrenamiento)")
ax.set_xlabel("Gama de precio (0 = más baja, 3 = más alta)")
ax.set_ylabel("Número de celulares")

plt.show()

Mínimos y máximos de cada característica

In [ ]:
X_train.describe().T[["min", "mean", "max"]].round(2)

Ceros en las columnas donde un cero puede no tener sentido

In [ ]:
(X_train[["px_height", "sc_w", "fc", "pc"]] == 0).sum()

In [ ]:
y_train[X_train["sc_w"] == 0].value_counts().sort_index() #gamas de los celulares con sc_w en 0

**Qué veo.** En entrenamiento no hay faltantes ni filas duplicadas, y las cuatro gamas tienen exactamente 400 celulares, así que no hace falta balancear nada.

Hay ceros que no pueden ser reales: 144 celulares con `sc_w` (ancho de pantalla) en 0 cm y 1 con `px_height` (alto de la pantalla en pixeles) en 0. Los ceros de `fc` (372) y `pc` (82) sí pueden pasar, porque son celulares sin cámara frontal o sin cámara principal, así que los dejo.

**Decisión:** igual que en el proyecto 1 con `chol` y `trestbps`, convierto esos ceros en faltantes y los relleno con la mediana dentro del `Pipeline`. No borro filas: son 144 de 1600 y los ceros de `sc_w` se reparten parejo entre las gamas (39, 39, 34 y 32), así que no parecen tener que ver con el precio. Cambiar un 0 por faltante es una regla fila por fila que no aprende nada de los datos, por eso la aplico igual en entrenamiento y en prueba.

In [ ]:
for col in ["px_height", "sc_w"]:
    X_train[col] = X_train[col].replace(0, np.nan)
    X_test[col] = X_test[col].replace(0, np.nan)

X_train[["px_height", "sc_w"]].isna().sum()

### 1.3 Mi predicción y el método de selección

**Antes de entrenar, las tres características que espero que más ayuden:**

1. `ram`: un celular con más memoria RAM normalmente cuesta más.
2. `int_memory`: el almacenamiento es de lo primero que se anuncia en la tienda y sube el precio.
3. `pc`: la cámara principal suele ser mejor en los celulares de gama alta.

**Método de selección: `SelectKBest` con `f_classif` (ANOVA)**, como en la clase 09. Para cada característica, por separado, compara qué tan distintos son sus promedios entre las cuatro gamas contra lo que varía dentro de cada gama. Esa razón es la puntuación F. Conserva las `k` características con mayor F. No mira combinaciones entre características.

El valor de `k` no lo fijo a ojo: pruebo 4, 6, 8 y 12 dentro de la validación cruzada.

Para entender el criterio, calculo a mano la F de `ram` y la comparo con `f_classif`.

In [ ]:
ram_por_gama = X_train["ram"].groupby(y_train)
media_total = X_train["ram"].mean()

entre_gamas = (ram_por_gama.size() * (ram_por_gama.mean() - media_total) ** 2).sum() / (4 - 1)
dentro_gamas = ((X_train["ram"] - ram_por_gama.transform("mean")) ** 2).sum() / (len(X_train) - 4)

print("F de ram a mano:", round(entre_gamas / dentro_gamas, 1))
print("F de ram con f_classif:", round(f_classif(X_train[["ram"]], y_train)[0][0], 1))

**Qué veo.** Sale lo mismo: 2832.6. La F de `ram` es tan grande porque su promedio cambia mucho de una gama a otra comparado con lo que varía dentro de cada gama.

## Etapa 2 · Entrenamiento, comparación y recomendación

1. Entrena **Random Forest, KNN, SVC y regresión logística**. Compara cada modelo usando las 20 características y usando el método de selección que elegiste.
2. Prueba al menos dos valores de un hiperparámetro por modelo. Usa las mismas tres particiones de validación cruzada para todos. En cada partición, ajusta el selector y el preprocesamiento solo con las filas de entrenamiento. Aplica escalado cuando el modelo lo requiera.
3. Presenta **una tabla de cuatro filas**, una por modelo, con la accuracy media de validación con todas las características y con selección. Indica cuáles conservó el selector al ajustarlo con todo el entrenamiento y compáralas con tu predicción inicial.
4. Elige una configuración a partir de la tabla y evalúala una vez en el 20 % de prueba. Reporta cuántos celulares clasificó correctamente y entre qué gamas se equivocó con mayor frecuencia.
5. Recomienda a la tienda un modelo y un conjunto de características. Justifica si vale la pena pedir menos especificaciones con base en tus resultados.

### 2.1 Modelos, rejillas y validación cruzada

Uso las mismas 3 particiones para los cuatro modelos. Cada modelo va dentro de un `Pipeline`, así la mediana, el escalado y el selector se ajustan solo con las filas de entrenamiento de cada partición. Escalo KNN, SVC y regresión logística, que trabajan con distancias o coeficientes; Random Forest no lo necesita porque solo compara cada característica contra un umbral.

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
valores_k = [4, 6, 8, 12]

In [ ]:
candidatos = [
    ("Random Forest", "bosque", RandomForestClassifier(n_estimators=100, max_features="sqrt", random_state=RANDOM_STATE),
     {"bosque__max_depth": [5, 10, None]}, False),
    ("KNN", "knn", KNeighborsClassifier(), {"knn__n_neighbors": [5, 15, 25]}, True),
    ("SVC", "svm", SVC(), {"svm__C": [0.1, 1, 10], "svm__kernel": ["linear", "rbf"]}, True),
    ("Regresión logística", "logistica", LogisticRegression(max_iter=2000), {"logistica__C": [0.1, 1, 10]}, True),
]

In [ ]:
resultados = []
busquedas = {}
for nombre, paso, modelo, rejilla, escalar in candidatos:
    for version in ["todas", "con selección"]:
        pasos = [("imputador", SimpleImputer(strategy="median"))]
        if escalar:
            pasos.append(("escala", StandardScaler()))
        rejilla_version = dict(rejilla)
        if version == "con selección":
            pasos.append(("seleccion", SelectKBest(score_func=f_classif)))
            rejilla_version["seleccion__k"] = valores_k
        pasos.append((paso, modelo))

        busqueda = GridSearchCV(Pipeline(pasos), rejilla_version, cv=cv, scoring="accuracy")
        busqueda.fit(X_train, y_train)
        busquedas[(nombre, version)] = busqueda
        resultados.append({"modelo": nombre, "versión": version,
                           "accuracy_cv": busqueda.best_score_, "parámetros": busqueda.best_params_})

### 2.2 Tabla de comparación

In [ ]:
tabla_resultados = pd.DataFrame(resultados)
tabla = tabla_resultados.pivot(index="modelo", columns="versión", values="accuracy_cv")
tabla = tabla.reindex(["Random Forest", "KNN", "SVC", "Regresión logística"])
tabla["diferencia"] = tabla["con selección"] - tabla["todas"]
tabla.round(4)

In [ ]:
tabla_resultados[["modelo", "versión", "parámetros"]]

**Qué veo.** La selección le ayuda a los cuatro modelos, pero no igual. KNN es el que más cambia: con las 20 características saca 0.5706 y con 4 sube a 0.8863. Random Forest sube de 0.8594 a 0.9062, también con 4. SVC y regresión logística casi no cambian (menos de un punto porcentual) y empatan arriba con 0.9631, los dos con `k = 12` y `C = 10`.

Para ver cuántas características conviene pedir, saco la mejor accuracy de cada modelo con cada valor de `k`. La columna 20 es usar todas.

In [ ]:
por_k = {}
for nombre, _, _, _, _ in candidatos:
    res = pd.DataFrame(busquedas[(nombre, "con selección")].cv_results_)
    por_k[nombre] = res.groupby("param_seleccion__k")["mean_test_score"].max()
    por_k[nombre][20] = busquedas[(nombre, "todas")].best_score_

pd.DataFrame(por_k).T.round(4)

**Qué veo.** KNN y Random Forest van mejor con pocas características: su mejor resultado es con 4 y, en general, bajan conforme agrego más. KNN baja de 0.8863 con 4 a 0.5706 con las 20. Como KNN mide distancias, las características que aportan poco pueden dificultar la clasificación. SVC y regresión logística hacen lo contrario, pero suben muy poco: la regresión logística pasa de 0.9544 con 4 a 0.9631 con 12.

### 2.3 Qué características conservó el selector

Ajusto la mediana y el selector con todo el entrenamiento. Uso `k=12`, el que eligieron SVC y regresión logística, y veo también cuáles serían las primeras 4.

In [ ]:
selector_todo = Pipeline([
    ("imputador", SimpleImputer(strategy="median")),
    ("seleccion", SelectKBest(score_func=f_classif, k=12)),
])
selector_todo.fit(X_train, y_train)

In [ ]:
puntuaciones = pd.Series(selector_todo.named_steps["seleccion"].scores_, index=X_train.columns)
puntuaciones.sort_values(ascending=False).round(1)

In [ ]:
conservadas_12 = X_train.columns[selector_todo.named_steps["seleccion"].get_support()].tolist()
primeras_4 = puntuaciones.sort_values(ascending=False).index[:4].tolist()
print("Con k=12:", conservadas_12)
print("Las primeras 4:", primeras_4)
print("Quita:", [c for c in X_train.columns if c not in conservadas_12])

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

puntuaciones.sort_values().plot.barh(ax=ax, logx=True)

ax.set_title("Puntuación F de cada característica (entrenamiento)")
ax.set_xlabel("Puntuación F (escala logarítmica)")

plt.tight_layout()
plt.show()

**Qué veo.** Con `k = 12` el selector quita `blue`, `clock_speed`, `dual_sim`, `fc`, `pc`, `sc_w`, `three_g` y `wifi`. Lo que más se nota es el salto en las puntuaciones: `ram` tiene 2832.6, luego vienen `battery_power` (25.7), `px_width` (17.6) y `px_height` (12.0), y de ahí para abajo todas tienen menos de 4. Según el criterio de ANOVA, estas 4 son las que destacan.

**Contra mi predicción:** acerté con `ram`, que es por mucho la más importante. `int_memory` sí entra en las 12, pero con una F de apenas 2.4, en el lugar 8. `pc` no entra: tiene 1.0. Las que no esperaba eran `battery_power` y la resolución de la pantalla (`px_width` y `px_height`).

### 2.4 Configuración elegida y prueba

Elijo **regresión logística con selección (`k = 12`, `C = 10`)**. Empata con SVC en la mejor accuracy de la tabla (0.9631) y con el mismo número de características, así que me quedo con el modelo más sencillo de los dos. La pruebo una sola vez con el 20% de prueba.

In [ ]:
modelo_final = busquedas[("Regresión logística", "con selección")].best_estimator_
modelo_final

In [ ]:
y_pred = modelo_final.predict(X_test)

In [ ]:
correctos = (y_pred == y_test).sum()
print("Clasificó bien", correctos, "de", len(y_test), "celulares")
print("Accuracy en prueba:", round(accuracy_score(y_test, y_pred), 4))

In [ ]:
print(classification_report(y_test, y_pred, digits=3))

In [ ]:
matriz = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots()

sns.heatmap(matriz, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)

ax.set_title("Regresión logística con 12 características · prueba")
ax.set_xlabel("Gama predicha")
ax.set_ylabel("Gama real")

plt.show()

**Qué veo.** Clasificó bien 394 de los 400 celulares de prueba (accuracy de 0.985). La gama 0 la acertó completa. Los 6 errores son entre gamas vecinas: 2 celulares de gama 1 que predijo como 2, 2 de gama 2 que predijo como 3 y 2 de gama 3 que predijo como 2. Donde más se equivoca es entre las gamas 2 y 3, con 4 de los 6 errores. Nunca confundió gamas que están a más de un paso, como 0 con 2.

---
## Recomendación para la tienda

**Modelo:** regresión logística con el selector ANOVA. En validación cruzada saca 0.9631 y en prueba acertó 394 de 400.

**¿Vale la pena pedir menos especificaciones?** Sí. Con solo 4 datos (`ram`, `battery_power`, `px_width` y `px_height`) la regresión logística saca 0.9544 en validación cruzada, contra 0.9575 con las 20 y 0.9631 con 12. Contra las 12 es menos de un punto porcentual de diferencia: de 1600 celulares, unos 14 más mal clasificados. Para la tienda es más fácil pedir 4 datos que 20, y la RAM, la batería y la resolución de la pantalla suelen venir en la ficha técnica. Si la tienda ya tiene la ficha completa, conviene usar las 12, que dan el resultado más alto.

## Decisiones

| # | Decisión | Elección | Por qué |
|---|---|---|---|
| 1 | Ceros en `sc_w` y `px_height` | Faltantes, rellenos con la mediana | Una pantalla de 0 cm no existe; los ceros se reparten parejo entre gamas |
| 2 | Ceros en `fc` y `pc` | Dejarlos | Un celular puede no tener cámara |
| 3 | Selección | `SelectKBest` con ANOVA, `k` dentro de la rejilla | El criterio se entiende y se ajusta dentro de cada partición |
| 4 | Escalado | Solo KNN, SVC y regresión logística | Usan distancias o coeficientes |
| 5 | Modelo | Regresión logística, `k = 12`, `C = 10` | Empata con SVC en 0.9631 y es el más sencillo |

## Limitaciones

- En prueba salió mejor que en validación cruzada (0.985 contra 0.9631). Con 400 celulares cada error mueve la accuracy 0.0025, así que para la tienda tomo como referencia la validación cruzada.
- ANOVA revisa cada característica por separado, así que no encuentra características que solo sirven juntas.